# Hotel booking cancellations

Question: which bookings will cancel, using only what the desk knows when the reservation is made?

Source: Hotel Booking Demand (Antonio, Almeida, and Nunes), two hotels in Portugal, arrivals from July 2015 through August 2017.

In [ ]:
import pandas as pd
from src.prepare import read_bookings
from src.paths import SOURCE_CSV
from src.split import split_bookings

bookings = read_bookings(SOURCE_CSV)
print(bookings.shape)
print(bookings['is_canceled'].mean())

(119390, 32)
0.37041628277075134


## Accuracy is the wrong headline

37.0% of bookings cancel, so a model that always says "will stay" is already 63.0% accurate. ROC-AUC and PR-AUC are the comparison metrics. Recall at the high-risk cutoff is what the desk acts on.

## `reservation_status` is the outcome

Check-Out lines up with stays. Canceled and No-Show line up with cancellations. Keeping this column, or the date it was written, produces a fake score near 99%.

In [ ]:
pd.crosstab(bookings['reservation_status'], bookings['is_canceled'])

is_canceled             0      1
reservation_status              
Canceled                0  43017
Check-Out           75166      0
No-Show                 0   1207

## Missing values are not all the same kind of gap

Company is empty on 94.3% of rows. That gap means "not a corporate booking," so it becomes a yes/no flag rather than an imputed id.

Agent is empty on 13.7% of rows. The id is a label, so the model gets a has-agent flag plus the training frequency of that id. Agent 240 is not a larger quantity than agent 9.

Country is missing on 0.41% of rows and has well over 100 distinct values. Full one-hot would add a sparse column per country. Frequency encoding and a top-20 plus Other encoding are both logged, and the holdout picks one.

Children is missing on 0.00% of rows and is treated as zero guests of that type.

In [ ]:
missing = bookings.isna().mean().sort_values(ascending=False)
print((missing[missing > 0] * 100).round(2))

company     94.31
agent       13.69
country      0.41
children     0.00


## Price and meal cleaning

The maximum ADR is 5400, against a 99th percentile of 252. That single outlier is capped at 1000. Rows with a negative ADR, and rows with zero guests, are dropped.

`meal` contains 1169 `Undefined` values. The dataset notes say Undefined and SC both mean no meal package, so they are merged to SC.

In [ ]:
print(bookings['adr'].describe())
print(bookings['meal'].value_counts(dropna=False))

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798


## Columns filled in after the booking

Assigned room differs from the reserved room on 12.5% of rows. That reassignment happens at check-in, not when the manager takes the booking.

When `booking_changes` is already above zero, the cancellation rate is 15.7%. When it is still zero, the rate is 40.9%. Changes accumulate during the life of the booking, so the column is not available at decision time. `days_in_waiting_list` has the same problem. A later experiment measures how much they inflate the score, then they are left out.

## The test set is a future summer

v1 is 2015–2016 (78,703 rows). v2 adds January–May 2017 (103,505 rows). June–August 2017 (15,885 rows, 39.3% canceled) is never used for fitting. Every model is scored on that same window.

`arrival_date_year` is not a model feature. A booking for a later year would otherwise be a number the model has never seen. The year is used only to derive the weekday, which is then encoded as sine and cosine, along with month and week, so December sits next to January.

## What the model is allowed to see

Dropped outright: reservation_status, reservation_status_date.

Tested, then dropped: assigned_room_type, booking_changes, days_in_waiting_list.

Engineered because each one has a desk meaning: total nights, total guests, a family flag, weekday of arrival, and the guest's past cancellation ratio.

PCA is not used. There are few numeric columns, they are different kinds of quantities, and a manager has to be able to say why a booking is risky.